# 📌 NOTE CHỤP HÌNH CHO BÁO CÁO
Các ghi chú có biểu tượng **📸 CHÈN HÌNH BÁO CÁO** đã được đặt trực tiếp ngay trên cell cần chụp. Dùng `Ctrl+F` và tìm cụm **CHÈN HÌNH BÁO CÁO** để nhảy nhanh giữa các vị trí.


# Decision Tree Regressor - From scratch and `scikit-learn` approach

### 1. Cấu trúc Node
* **Decision Node:** Lưu `feature` và `threshold` để chia dữ liệu, kèm con trỏ `left`, `right`. Thuộc tính `value = None`.
* **Leaf Node:** Chỉ lưu `value` (giá trị dự đoán). Các thuộc tính `feature`, `threshold`, `left`, `right` đều bằng `None`.

### 2. Các bước thực hiện

* **Bước 1 (Khởi tạo):** Bắt đầu tại Root Node với toàn bộ tập dữ liệu `Training`.
* **Bước 2 (Chọn feature và threshold):** 
  * Duyệt qua các feature và các ngưỡng giá trị (ngưỡng thường là trung bình của các giá trị liên tiếp sau khi sắp xếp).
  * Thử chia dữ liệu và tính weighted MSE của 2 nhánh. Cặp `(feature, threshold)` nào làm **tối thiểu hóa weighted MSE** sẽ được chọn.
* **Bước 3 (Chia node con):** Chia tập dữ liệu hiện tại thành 2 tập con vật lý:
  * **Nhánh bên trái (`left`):** Chứa các dữ liệu có giá trị feature $\le \text{threshold}$.
  * **Nhánh bên phải (`right`):** Chứa các dữ liệu có giá trị feature $> \text{threshold}$.
* **Bước 4 (Đệ quy):** Lặp lại Bước 2 và Bước 3 cho các node `left` và `right` cho đến khi đạt điều kiện dừng (ví dụ: đạt `max_depth`, `samples` khi chia quá ít).
* **Bước 5 (Gán output tại Leaf):** Khi dừng, biến node đó thành Node lá. Gán `value` bằng **Mean** của tập dữ liệu tại node lá đó; gán `left = None` và `right = None`.

## Triển khai `DTR` không dùng thư viện

Import thư viện cần thiết trước khi code

In [1]:

from __future__ import annotations

import pandas as pd 
import numpy as np
import joblib
import sklearn.tree, time
from dataclasses import dataclass
import os, sys


sys.path.insert(0, os.path.abspath('.'))
from utils.custom_hyperparameter_tuning import CustomGridSearchCV
from utils.custom_cv import CustomKFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.tree import DecisionTreeRegressor

from pathlib import Path

# Tìm thư mục gốc của Practice 2 để import utils và đọc data bằng đường dẫn tương đối.
PROJECT_ROOT = Path.cwd().resolve()
if not ((PROJECT_ROOT / 'utils').exists() and (PROJECT_ROOT / 'data').exists()):
    for candidate in PROJECT_ROOT.parents:
        if (candidate / 'utils').exists() and (candidate / 'data').exists():
            PROJECT_ROOT = candidate
            break
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.custom_hyperparameter_tuning import CustomGridSearchCV
from utils.custom_cv import CustomKFold
from utils.model_manager import save_model_package


In [2]:
X_train = joblib.load('./data/ready_for_train/X_train_final.pkl')
X_test = joblib.load('./data/ready_for_train/X_test_final.pkl')

y_train_log = joblib.load('./data/ready_for_train/y_train_log.pkl')
y_test_log = joblib.load('./data/ready_for_train/y_test_log.pkl')
# Previewing Train, test shapes

print('Training dataset shape:')
print(f'X: {X_train.shape}')
print(f'y: {y_train_log.shape}')

print('Testing dataset shape:')
print(f'X: {X_test.shape}')
print(f'y: {y_test_log.shape}')

Training dataset shape:
X: (800, 11)
y: (800,)
Testing dataset shape:
X: (200, 11)
y: (200,)


Triển khai Decision Tree Regressor không dùng `sklearn`

---
> 📸 **HÌNH 26 — CHÈN HÌNH BÁO CÁO [05-TRAIN-DECISION-TREE-REGRESSOR-01] — Mục 5.2.2 – Decision Tree From Scratch**  
> Chụp phần Node và các hàm xây cây/tìm split trong cell code ngay bên dưới. Không cần chụp toàn bộ nếu quá dài.  
> **Vị trí:** cell code nằm **ngay bên dưới ghi chú này**.
---

In [3]:
@dataclass
class Node:
    feature: int | None = None
    threshold: float | None = None
    left: "Node | None" = None
    right: "Node | None" = None
    value: float | None = None

    def is_leaf_node(self) -> bool:
        return self.value is not None


class DTR:
    def __init__(self, max_depth: int = 5, min_samples_split: int = 3, min_samples_leaf: int = 2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.root: Node | None = None

    def __mse(self, y_actual: np.ndarray, y_hat: float) -> float:
        """Tính Mean Squared Error giữa y thực tế và một giá trị dự đoán.

        Args:
            y_actual (np.ndarray): Mảng target thực tế.
            y_hat (float): Giá trị dự đoán đại diện cho node.

        Returns:
            float: Giá trị MSE.
        """
        residual = y_actual - y_hat
        return float(np.mean(residual ** 2))

    def set_params(self, **params) -> "DTR":
        for key, value in params.items():
            setattr(self, key, value)
        return self

    def get_params(self) -> dict:
        return {
            "max_depth": self.max_depth,
            "min_samples_split": self.min_samples_split,
            "min_samples_leaf": self.min_samples_leaf,
        }

    def fit(self, X: np.ndarray, y: np.ndarray) -> "DTR":
        """Train model Decision Tree Regressor với tập input là dữ liệu dạng NumPy.

        Args:
            X (np.ndarray): Ma trận feature có shape `(n_samples, n_features)`.
            y (np.ndarray): Vector target có shape `(n_samples,)`.

        Returns:
            DTR: Mô hình sau khi đã xây cây.
        """
        X = np.asarray(X)
        y = np.asarray(y).ravel()
        self.root = self.__build_tree(X, y, 0)
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Dự đoán target cho dữ liệu đầu vào dạng NumPy.

        Args:
            X (np.ndarray): Ma trận feature có shape `(n_samples, n_features)`.

        Returns:
            np.ndarray: Vector dự đoán có shape `(n_samples,)`.
        """
        X = np.asarray(X)
        return np.array([self.__traverse_tree(row, self.root) for row in X])

    def __traverse_tree(self, x: np.ndarray, node: Node) -> float:
        if node.is_leaf_node():
            return node.value

        if x[node.feature] <= node.threshold:
            return self.__traverse_tree(x, node.left)

        return self.__traverse_tree(x, node.right)

    def __build_tree(self, X: np.ndarray, y: np.ndarray, depth: int) -> Node:
        n_samples = X.shape[0]

        if (depth >= self.max_depth
            or n_samples < self.min_samples_split
            or n_samples < 2 * self.min_samples_leaf
            or len(np.unique(y)) == 1):
            return Node(value=float(np.mean(y)))

        best_feature, best_threshold = self._get_best_split_criteria(X, y)

        if best_feature is None:
            return Node(value=float(np.mean(y)))

        left_mask = X[:, best_feature] <= best_threshold

        X_left = X[left_mask]
        y_left = y[left_mask]

        X_right = X[~left_mask]
        y_right = y[~left_mask]

        left_child = self.__build_tree(X_left, y_left, depth + 1)
        right_child = self.__build_tree(X_right, y_right, depth + 1)

        return Node(
            feature=best_feature,
            threshold=best_threshold,
            left=left_child,
            right=right_child,
            value=None,
        )

    def _get_best_split_criteria(self, X: np.ndarray, y: np.ndarray) -> tuple[int, float] | tuple[None, None]:
        """Tìm feature index và threshold tốt nhất để chia node hiện tại.
        (Feature index tức là ta sẽ thu thập index của best feature đó)
        Args:
            X (np.ndarray): Ma trận feature của node hiện tại, shape
                `(n_samples, n_features)`.
            y (np.ndarray): Vector target của node hiện tại, shape `(n_samples,)`.

        Returns:
            tuple[int, float] | tuple[None, None]: Cặp `(feature_index, threshold)`
            tốt nhất. Nếu không tìm được split hợp lệ, trả về `(None, None)`.

        Thuật toán chính:
            1. Tính MSE của node hiện tại làm mốc lỗi ban đầu.
            2. Duyệt qua từng feature index trong X.
            3. Với mỗi feature, tạo candidate threshold bằng midpoint giữa các
               giá trị unique liên tiếp sau khi sắp xếp.
            4. Với mỗi threshold, chia y thành nhánh trái/phải bằng boolean mask.
            5. Bỏ qua split nếu một nhánh có ít mẫu hơn `min_samples_leaf`.
            6. Tính weighted MSE của split và cập nhật split tốt nhất nếu lỗi nhỏ hơn.
        """
        current_mse = self.__mse(y, float(np.mean(y)))
        best_feature = None
        best_threshold = None
        n_features = X.shape[1]

        for feature in range(n_features):
            x_arr = X[:, feature]
            unique_vals = np.sort(np.unique(x_arr))
            splits = (unique_vals[:-1] + unique_vals[1:]) / 2.0

            for split in splits:
                left_mask = x_arr <= split

                n_left = np.sum(left_mask)
                n_right = len(y) - n_left

                if n_left < self.min_samples_leaf or n_right < self.min_samples_leaf:
                    continue

                y_left = y[left_mask]
                y_right = y[~left_mask]

                left_mse = self.__mse(y_left, float(np.mean(y_left)))
                right_mse = self.__mse(y_right, float(np.mean(y_right)))
                weighted_mse = ((left_mse * n_left) + (right_mse * n_right)) / len(y)

                if weighted_mse < current_mse:
                    current_mse = weighted_mse
                    best_feature = feature
                    best_threshold = float(split)

        return (best_feature, best_threshold)

### Kiểm tra nhanh mô hình Decision Tree From Scratch


In [4]:
y_train_original = np.expm1(y_train_log)
y_test_original  = np.expm1(y_test_log)

# Quick test để kiểm tra chức năng — train trên y_log, đánh giá trên thang đo gốc
quick_dtr = DTR()

start = time.time()
quick_dtr.fit(X_train, y_train_log.values)
train_time = time.time() - start

quick_pred = np.expm1(quick_dtr.predict(X_test))
r2   = r2_score(y_test_original, quick_pred)
rmse = np.sqrt(mean_squared_error(y_test_original, quick_pred))
mae  = mean_absolute_error(y_test_original, quick_pred)
mape = mean_absolute_percentage_error(y_test_original, quick_pred) * 100

print(f'Quick test - Train time: {train_time:.1f}s')
print(f'R²   : {r2:.4f}')
print(f'RMSE : {rmse:.2f}')
print(f'MAE  : {mae:.2f}')
print(f"MAPE : {mape:.2f} %")

train_pred_original = np.expm1(
    quick_dtr.predict(X_train)
)

Quick test - Train time: 0.1s
R²   : 0.6079
RMSE : 338.80
MAE  : 197.37
MAPE : 55.74 %


---
> 📸 **HÌNH 27A (TÙY CHỌN) — CHÈN HÌNH BÁO CÁO [05-TRAIN-DECISION-TREE-REGRESSOR-02] — Mục 5.3 – Hyperparameter Grid của Decision Tree**  
> Chụp cell code ngay bên dưới: max_depth, min_samples_split và min_samples_leaf.  
> **Vị trí:** cell code nằm **ngay bên dưới ghi chú này**.
---

## Hyperparameter Tuning

Grid Search kết hợp với **5-fold Cross Validation** được sử dụng trên tập huấn luyện gồm **800 mẫu**.

```python
dtr_grid = {
    'max_depth': [3, 5, 8, 12],
    'min_samples_split': [100, 500, 1000],
    'min_samples_leaf': [50, 100, 500]
}
```

Mô hình được huấn luyện trên `log1p(Total Amount)` và được đánh giá lại trên thang `Total Amount` gốc bằng `np.expm1()`. Kết quả tốt nhất sau khi chạy lại là:

```python
{
    'max_depth': 3,
    'min_samples_split': 100,
    'min_samples_leaf': 100
}
```


---
> 📸 **HÌNH 27A (TÙY CHỌN) — CHÈN HÌNH BÁO CÁO [05-TRAIN-DECISION-TREE-REGRESSOR-03] — Mục 5.3 – Hyperparameter Grid của Decision Tree**  
> Chụp cell code ngay bên dưới: max_depth, min_samples_split và min_samples_leaf.  
> **Vị trí:** cell code nằm **ngay bên dưới ghi chú này**.
---

In [5]:
dtr_grid = {'max_depth': [3,5,8,12],
            'min_samples_split': [100,500,1000],
            'min_samples_leaf': [50,100,500]}

### Grid Search dùng Neg RMSE

In [6]:
cv = CustomKFold(n_splits=5, shuffle=True, random_state=42)
scratch_dtr = DTR()

dtr_grid_search = CustomGridSearchCV(estimator=scratch_dtr,
                                     param_grid = dtr_grid,
                                     cv=cv,scoring = 'neg_rmse')


dtr_grid_search.fit(X_train, y_train_log.values)

Bắt đầu GridSearchCV: 36 tổ hợp tham số, 5 folds.


[1/36] Params: {'max_depth': 3, 'min_samples_split': 100, 'min_samples_leaf': 50} --> neg_rmse: -0.5220
[2/36] Params: {'max_depth': 3, 'min_samples_split': 100, 'min_samples_leaf': 100} --> neg_rmse: -0.5191
[3/36] Params: {'max_depth': 3, 'min_samples_split': 100, 'min_samples_leaf': 500} --> neg_rmse: -1.3535
[4/36] Params: {'max_depth': 3, 'min_samples_split': 500, 'min_samples_leaf': 50} --> neg_rmse: -0.5819
[5/36] Params: {'max_depth': 3, 'min_samples_split': 500, 'min_samples_leaf': 100} --> neg_rmse: -0.5819


[6/36] Params: {'max_depth': 3, 'min_samples_split': 500, 'min_samples_leaf': 500} --> neg_rmse: -1.3535
[7/36] Params: {'max_depth': 3, 'min_samples_split': 1000, 'min_samples_leaf': 50} --> neg_rmse: -1.3535
[8/36] Params: {'max_depth': 3, 'min_samples_split': 1000, 'min_samples_leaf': 100} --> neg_rmse: -1.3535
[9/36] Params: {'max_depth': 3, 'min_samples_split': 1000, 'min_samples_leaf': 500} --> neg_rmse: -1.3535


[10/36] Params: {'max_depth': 5, 'min_samples_split': 100, 'min_samples_leaf': 50} --> neg_rmse: -0.5248
[11/36] Params: {'max_depth': 5, 'min_samples_split': 100, 'min_samples_leaf': 100} --> neg_rmse: -0.5191
[12/36] Params: {'max_depth': 5, 'min_samples_split': 100, 'min_samples_leaf': 500} --> neg_rmse: -1.3535


[13/36] Params: {'max_depth': 5, 'min_samples_split': 500, 'min_samples_leaf': 50} --> neg_rmse: -0.5819
[14/36] Params: {'max_depth': 5, 'min_samples_split': 500, 'min_samples_leaf': 100} --> neg_rmse: -0.5819
[15/36] Params: {'max_depth': 5, 'min_samples_split': 500, 'min_samples_leaf': 500} --> neg_rmse: -1.3535
[16/36] Params: {'max_depth': 5, 'min_samples_split': 1000, 'min_samples_leaf': 50} --> neg_rmse: -1.3535
[17/36] Params: {'max_depth': 5, 'min_samples_split': 1000, 'min_samples_leaf': 100} --> neg_rmse: -1.3535
[18/36] Params: {'max_depth': 5, 'min_samples_split': 1000, 'min_samples_leaf': 500} --> neg_rmse: -1.3535


[19/36] Params: {'max_depth': 8, 'min_samples_split': 100, 'min_samples_leaf': 50} --> neg_rmse: -0.5248
[20/36] Params: {'max_depth': 8, 'min_samples_split': 100, 'min_samples_leaf': 100} --> neg_rmse: -0.5191
[21/36] Params: {'max_depth': 8, 'min_samples_split': 100, 'min_samples_leaf': 500} --> neg_rmse: -1.3535


[22/36] Params: {'max_depth': 8, 'min_samples_split': 500, 'min_samples_leaf': 50} --> neg_rmse: -0.5819
[23/36] Params: {'max_depth': 8, 'min_samples_split': 500, 'min_samples_leaf': 100} --> neg_rmse: -0.5819
[24/36] Params: {'max_depth': 8, 'min_samples_split': 500, 'min_samples_leaf': 500} --> neg_rmse: -1.3535
[25/36] Params: {'max_depth': 8, 'min_samples_split': 1000, 'min_samples_leaf': 50} --> neg_rmse: -1.3535
[26/36] Params: {'max_depth': 8, 'min_samples_split': 1000, 'min_samples_leaf': 100} --> neg_rmse: -1.3535
[27/36] Params: {'max_depth': 8, 'min_samples_split': 1000, 'min_samples_leaf': 500} --> neg_rmse: -1.3535


[28/36] Params: {'max_depth': 12, 'min_samples_split': 100, 'min_samples_leaf': 50} --> neg_rmse: -0.5248
[29/36] Params: {'max_depth': 12, 'min_samples_split': 100, 'min_samples_leaf': 100} --> neg_rmse: -0.5191
[30/36] Params: {'max_depth': 12, 'min_samples_split': 100, 'min_samples_leaf': 500} --> neg_rmse: -1.3535


[31/36] Params: {'max_depth': 12, 'min_samples_split': 500, 'min_samples_leaf': 50} --> neg_rmse: -0.5819
[32/36] Params: {'max_depth': 12, 'min_samples_split': 500, 'min_samples_leaf': 100} --> neg_rmse: -0.5819
[33/36] Params: {'max_depth': 12, 'min_samples_split': 500, 'min_samples_leaf': 500} --> neg_rmse: -1.3535
[34/36] Params: {'max_depth': 12, 'min_samples_split': 1000, 'min_samples_leaf': 50} --> neg_rmse: -1.3535
[35/36] Params: {'max_depth': 12, 'min_samples_split': 1000, 'min_samples_leaf': 100} --> neg_rmse: -1.3535
[36/36] Params: {'max_depth': 12, 'min_samples_split': 1000, 'min_samples_leaf': 500} --> neg_rmse: -1.3535

-> Tham số TỐT NHẤT: {'max_depth': 3, 'min_samples_split': 100, 'min_samples_leaf': 100}
-> Điểm neg_rmse TỐT NHẤT: -0.5191


### Grid Search dùng R2

In [7]:
cv = CustomKFold(n_splits=5, shuffle=True, random_state=42)
scratch_dtr = DTR()

dtr_grid_search = CustomGridSearchCV(estimator=scratch_dtr,param_grid = dtr_grid,cv=cv,scoring = 'r2')
dtr_grid_search.fit(X_train, y_train_log)

Bắt đầu GridSearchCV: 36 tổ hợp tham số, 5 folds.


[1/36] Params: {'max_depth': 3, 'min_samples_split': 100, 'min_samples_leaf': 50} --> r2: 0.8492


[2/36] Params: {'max_depth': 3, 'min_samples_split': 100, 'min_samples_leaf': 100} --> r2: 0.8509
[3/36] Params: {'max_depth': 3, 'min_samples_split': 100, 'min_samples_leaf': 500} --> r2: -0.0130
[4/36] Params: {'max_depth': 3, 'min_samples_split': 500, 'min_samples_leaf': 50} --> r2: 0.8124


[5/36] Params: {'max_depth': 3, 'min_samples_split': 500, 'min_samples_leaf': 100} --> r2: 0.8124
[6/36] Params: {'max_depth': 3, 'min_samples_split': 500, 'min_samples_leaf': 500} --> r2: -0.0130
[7/36] Params: {'max_depth': 3, 'min_samples_split': 1000, 'min_samples_leaf': 50} --> r2: -0.0130
[8/36] Params: {'max_depth': 3, 'min_samples_split': 1000, 'min_samples_leaf': 100} --> r2: -0.0130
[9/36] Params: {'max_depth': 3, 'min_samples_split': 1000, 'min_samples_leaf': 500} --> r2: -0.0130


[10/36] Params: {'max_depth': 5, 'min_samples_split': 100, 'min_samples_leaf': 50} --> r2: 0.8476


...

[20/36] Params: {'max_depth': 8, 'min_samples_split': 100, 'min_samples_leaf': 100} --> r2: 0.8509
[21/36] Params: {'max_depth': 8, 'min_samples_split': 100, 'min_samples_leaf': 500} --> r2: -0.0130


[22/36] Params: {'max_depth': 8, 'min_samples_split': 500, 'min_samples_leaf': 50} --> r2: 0.8124
[23/36] Params: {'max_depth': 8, 'min_samples_split': 500, 'min_samples_leaf': 100} --> r2: 0.8124
[24/36] Params: {'max_depth': 8, 'min_samples_split': 500, 'min_samples_leaf': 500} --> r2: -0.0130
[25/36] Params: {'max_depth': 8, 'min_samples_split': 1000, 'min_samples_leaf': 50} --> r2: -0.0130
[26/36] Params: {'max_depth': 8, 'min_samples_split': 1000, 'min_samples_leaf': 100} --> r2: -0.0130
[27/36] Params: {'max_depth': 8, 'min_samples_split': 1000, 'min_samples_leaf': 500} --> r2: -0.0130
[28/36] Params: {'max_depth': 12, 'min_samples_split': 100, 'min_samples_leaf': 50} --> r2: 0.8476


[29/36] Params: {'max_depth': 12, 'min_samples_split': 100, 'min_samples_leaf': 100} --> r2: 0.8509
[30/36] Params: {'max_depth': 12, 'min_samples_split': 100, 'min_samples_leaf': 500} --> r2: -0.0130


[31/36] Params: {'max_depth': 12, 'min_samples_split': 500, 'min_samples_leaf': 50} --> r2: 0.8124


[32/36] Params: {'max_depth': 12, 'min_samples_split': 500, 'min_samples_leaf': 100} --> r2: 0.8124
[33/36] Params: {'max_depth': 12, 'min_samples_split': 500, 'min_samples_leaf': 500} --> r2: -0.0130
[34/36] Params: {'max_depth': 12, 'min_samples_split': 1000, 'min_samples_leaf': 50} --> r2: -0.0130
[35/36] Params: {'max_depth': 12, 'min_samples_split': 1000, 'min_samples_leaf': 100} --> r2: -0.0130
[36/36] Params: {'max_depth': 12, 'min_samples_split': 1000, 'min_samples_leaf': 500} --> r2: -0.0130

-> Tham số TỐT NHẤT: {'max_depth': 3, 'min_samples_split': 100, 'min_samples_leaf': 100}
-> Điểm r2 TỐT NHẤT: 0.8509


## So sánh DTR không dùng thư viện và DTR dùng `sklearn`


---
> 📸 **HÌNH 27 — CHÈN HÌNH BÁO CÁO [05-TRAIN-DECISION-TREE-REGRESSOR-04] — Mục 5.3 và 6.2 – Kết quả Decision Tree**  
> Chụp Best params và output so sánh Decision Tree From Scratch với sklearn trong cell này.  
> **Vị trí:** cell code nằm **ngay bên dưới ghi chú này**.
---

In [8]:
best_params = dtr_grid_search.best_params_
print('Best params (neg_rmse):', best_params)

# ── From Scratch với best params ──────────────────────
scratch_best = dtr_grid_search.best_estimator_

start = time.time()
scratch_best.fit(X_train, y_train_log.values)
train_time_scratch = time.time() - start

scratch_pred = np.expm1(scratch_best.predict(X_test))

mae_scratch  = mean_absolute_error(y_test_original, scratch_pred)
rmse_scratch = np.sqrt(mean_squared_error(y_test_original, scratch_pred))
r2_scratch   = r2_score(y_test_original, scratch_pred)
mape_scratch = mean_absolute_percentage_error(y_test_original, scratch_pred) * 100
# ── sklearn với cùng cách tiếp cận (train trên y_log) ─ đánh giá trên y_test gốc
start = time.time()
sklearn_best = DecisionTreeRegressor(**best_params, random_state=42)
sklearn_best.fit(X_train, y_train_log)
train_time_sklearn = time.time() - start

sklearn_pred_best = np.expm1(sklearn_best.predict(X_test))

mae_sklearn  = mean_absolute_error(y_test_original, sklearn_pred_best)
rmse_sklearn = np.sqrt(mean_squared_error(y_test_original, sklearn_pred_best))
r2_sklearn   = r2_score(y_test_original, sklearn_pred_best)
mape_sklearn = mean_absolute_percentage_error(y_test_original, sklearn_pred_best) * 100
# ── Bảng so sánh ─────────────────────────────────────
print('\n' + '='*60)
print(f"{'Metric':<15} {'From Scratch':>20} {'sklearn':>20}")
print('='*60)
print(f"{'R²':<15} {r2_scratch:>20.4f} {r2_sklearn:>20.4f}")
print(f"{'RMSE':<15} {rmse_scratch:>20.2f} {rmse_sklearn:>20.2f}")
print(f"{'MAE':<15} {mae_scratch:>20.2f} {mae_sklearn:>20.2f}")
print(f"{'MAPE':<15} {mape_scratch:>20.2f} {mape_sklearn:>20.2f}")
print(f"{'Train Time (s)':<15} {train_time_scratch:>20.1f} {train_time_sklearn:>20.1f}")
print('='*60)

# So sánh sai lệch dự đoán
max_diff = np.max(np.abs(scratch_pred - sklearn_pred_best))
print(f'\nSai lệch dự đoán tối đa giữa 2 mô hình: {max_diff:.4f}')

Best params (neg_rmse): {'max_depth': 3, 'min_samples_split': 100, 'min_samples_leaf': 100}

Metric                  From Scratch              sklearn
R²                            0.7172               0.7172
RMSE                          287.74               287.74
MAE                           181.87               181.87
MAPE                           52.75                52.75
Train Time (s)                   0.0                  0.0

Sai lệch dự đoán tối đa giữa 2 mô hình: 0.0000


---
> 📸 **HÌNH 27B (TÙY CHỌN) — CHÈN HÌNH BÁO CÁO [05-TRAIN-DECISION-TREE-REGRESSOR-05] — Mục 6.2 – So sánh Decision Tree trên target gốc và log**  
> Chỉ chụp output cell này khi báo cáo có nhận xét về ảnh hưởng của Log Transform.  
> **Vị trí:** cell code nằm **ngay bên dưới ghi chú này**.
---

In [9]:

# ── From Scratch: y GỐC và y LOG ────────────────────────────────────
sc_orig = DTR(max_depth = 8, min_samples_split = 1000, min_samples_leaf = 50)
sc_orig.fit(X_train, y_train_original.values)
pred_sc_orig = sc_orig.predict(X_test)

sc_log = DTR(max_depth = 8, min_samples_split = 1000, min_samples_leaf = 50)
sc_log.fit(X_train, y_train_log.values)
pred_sc_log = np.expm1(sc_log.predict(X_test))

# ── sklearn DecisionTreeRegressor: y GỐC và y LOG ────────────────────────
sk_orig = sklearn.tree.DecisionTreeRegressor(max_depth = 8, min_samples_split = 1000, min_samples_leaf = 50)
sk_orig.fit(X_train, y_train_original)
pred_sk_orig = sk_orig.predict(X_test)

sk_log = sklearn.tree.DecisionTreeRegressor(max_depth = 8, min_samples_split = 1000, min_samples_leaf = 50)
sk_log.fit(X_train, y_train_log)
pred_sk_log = np.expm1(sk_log.predict(X_test))

# ── Bảng so sánh tổng hợp (đánh giá trên thang đo giá GỐC) ─────────
W = 18
print(f"{'Metric':<15} {'Scratch/y_gốc':>{W}} {'Scratch/y_log':>{W}} {'sklearn/y_gốc':>{W}} {'sklearn/y_log':>{W}}")
print('-' * (15 + W*4 + 3))
for label, pred in [('R²', None), ('RMSE', None), ('MAE', None)]:
    v1 = r2_score(y_test_original, pred_sc_orig)    if label == 'R²' else (np.sqrt(mean_squared_error(y_test_original, pred_sc_orig)) if label == 'RMSE' else mean_absolute_error(y_test_original, pred_sc_orig))
    v2 = r2_score(y_test_original, pred_sc_log)     if label == 'R²' else (np.sqrt(mean_squared_error(y_test_original, pred_sc_log))  if label == 'RMSE' else mean_absolute_error(y_test_original, pred_sc_log))
    v3 = r2_score(y_test_original, pred_sk_orig)    if label == 'R²' else (np.sqrt(mean_squared_error(y_test_original, pred_sk_orig)) if label == 'RMSE' else mean_absolute_error(y_test_original, pred_sk_orig))
    v4 = r2_score(y_test_original, pred_sk_log)     if label == 'R²' else (np.sqrt(mean_squared_error(y_test_original, pred_sk_log))  if label == 'RMSE' else mean_absolute_error(y_test_original, pred_sk_log))
    fmt = '.4f' if label == 'R²' else '.2f'
    print(f"{label:<15} {v1:>{W}{fmt}} {v2:>{W}{fmt}} {v3:>{W}{fmt}} {v4:>{W}{fmt}}")


Metric               Scratch/y_gốc      Scratch/y_log      sklearn/y_gốc      sklearn/y_log
------------------------------------------------------------------------------------------
R²                         -0.0003            -0.2493            -0.0003            -0.2493
RMSE                        541.13             604.73             541.13             604.73
MAE                         448.01             384.31             448.01             384.31


## Kết luận

### Hướng triển khai Decision Tree Regressor:
- From Scratch: thuật toán xây cây dựa trên việc duyệt qua các feature, thử các threshold có thể chia, tính weighted MSE cho từng cách chia và chọn split sao cho `MSE` là tối thiểu. Cách triển khai này giúp hiểu rõ hơn cơ chế hoạt động bên trong của Decision Tree, đặc biệt là cách mô hình chọn feature, threshold và tạo leaf node.
- Sử dụng `DecisionTreeRegressor` có sẵn từ `sklearn.tree`.


### So sánh hiệu năng giữa hai cách triển khai


Khi so sánh với `DecisionTreeRegressor` của `sklearn`, hai mô hình cho kết quả gần như giống hệt nhau trên tập test. Các metric như `R²`, `MAE`, `RMSE` và `MSE` gần như trùng nhau, trong khi sai khác lớn nhất giữa các giá trị dự đoán chỉ ở mức rất nhỏ do sai số tính toán số thực. Điều này cho thấy phiên bản Decision Tree Regressor tự triển khai đã mô phỏng khá sát logic cơ bản của thư viện `sklearn`.


### Hiệu năng thực sự của mô hình 


Kết quả `R²` đạt khoảng `0.73`, cho thấy mô hình chỉ giải thích được khoảng 73% sự biến thiên của `prices`. Đây là một mức kết quả tạm chấp nhận được.


### Hướng giải quyết tiếp theo


Nhìn chung, Decision Tree Regressor phù hợp để làm baseline model vì dễ hiểu, dễ giải thích và giúp quan sát được cách dữ liệu được chia theo từng feature. Tuy nhiên, để cải thiện hiệu năng dự đoán, các mô hình ensemble như Random Forest Regressor hoặc Gradient Boosting Regressor nên được thử nghiệm tiếp theo. Random Forest có thể giảm variance bằng cách kết hợp nhiều cây quyết định, trong khi Gradient Boosting có thể cải thiện dần sai số qua nhiều mô hình yếu liên tiếp.

## Lưu model


In [10]:
metrics_DTR_regression = {
    'R2': r2_scratch,
    'MAE': mae_scratch,
    'RMSE': rmse_scratch,
    'MAPE': mape_scratch
}

save_model_package(
    model=scratch_best,
    model_name="decision_tree_scratch",
    best_params=best_params,
    metrics=metrics_DTR_regression,
    save_dir='./models',
    extra={'target': 'log1p(Total Amount)'}
)

Đã lưu mô hình: models/decision_tree_scratch.pkl
Đã lưu kết quả: models/decision_tree_scratch_metrics.json


'models/decision_tree_scratch.pkl'